# Data Pre-Processing
## Description
Before we start directly working with the models, we load in all the files needed for the pre-processing. Among them are: matching.csv, which will be used to find tags that both book and movie datasets share; book-genome and movie tagdl.csv, which include lists of tags attributed to each book/movie; metadata_updated.json, which includes information about each movie's title, directors, main cast, average rating, imdb movie id, and item id; finally, we'll use ratings.json for both books and movies datasets, as they will be filtered by years we chose based on the data retention rate.

In the following pre-processing steps, we filter both book and movie datasets by their shared tags to ensure consistency throughout our data, and then apply a time period filter to movie datasets. The years used in the filter were chosen with the goal to both reduce the information scope and maintain the highest possible retention rate.

## Set-Up

In [2]:
# Fixed version for your environment
import os

import findspark
findspark.init("/usr/local/spark/")

In [3]:
# Import the SparkSession module
from pyspark.sql import SparkSession

# Import the collections module
import collections

# Create or get existing SparkSession
# Removed the hardcoded local[8] and let Spark decide based on your system
spark = SparkSession.builder \
    .appName("Project") \
    .config("spark.sql.repl.eagerEval.enabled", "true") \
    .config("spark.sql.repl.eagerEval.maxNumRows", 10) \
    .getOrCreate()

# Get SparkContext from the session
sc = spark.sparkContext

In [4]:
# Imports for content-based filtering
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, FloatType
from pyspark.sql.functions import explode, col, min as spark_min, max as spark_max
from pyspark.sql.functions import regexp_extract, when, trim
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from pprint import pprint

In [5]:
# Imports for collaborative filtering
from pyspark.sql.functions import col
from pyspark.ml.recommendation import ALS
from pyspark.sql.functions import udf

In [6]:
# Imports for combining 2 models
from numpy import dot
from numpy.linalg import norm
from collections import defaultdict
from pyspark.ml.linalg import Vectors
from pyspark.ml.linalg import DenseVector

## 0. Loading the files

In [7]:
# Load the matching.csv file from book-genome dataset
try:
    matching_df = spark.read.csv("./raw_data/book_dataset/matching.csv", header=True, inferSchema=True)
    print(f"Matching.csv shape: {matching_df.count()} rows, {len(matching_df.columns)} columns")
except Exception as e:
    print(f"Error loading matching.csv: {e}")
    matching_df = None

# Load tagdl.csv from book-genome dataset
try:
    book_tagdl_df = spark.read.csv("./raw_data/book_dataset/scores/tagdl.csv", header=True, inferSchema=True)
    print(f"Book tagdl.csv shape: {book_tagdl_df.count()} rows, {len(book_tagdl_df.columns)} columns")
except Exception as e:
    print(f"Error loading book tagdl.csv: {e}")
    book_tagdl_df = None

# Load tagdl.csv from movies dataset
try:
    movie_tagdl_df = spark.read.csv("./raw_data/movie_dataset_public_final/scores/tagdl.csv", header=True, inferSchema=True)
    print(f"Movie tagdl.csv shape: {movie_tagdl_df.count()} rows, {len(movie_tagdl_df.columns)} columns")
except Exception as e:
    print(f"Error loading movie tagdl.csv: {e}")
    movie_tagdl_df = None

# Load metadata_updated.json
try:
    metadata_df = spark.read.json("./raw_data/movie_dataset_public_final/raw/metadata_updated.json")
    print(f"Metadata shape: {metadata_df.count()} rows, {len(metadata_df.columns)} columns")
    metadata_df.show(5)
except Exception as e:
    print(f"Error loading metadata_updated.json: {e}")
    metadata_df = None

# Load movie ratings  
try:
    movie_ratings_df = spark.read.json("./raw_data/movie_dataset_public_final/raw/ratings.json")
    print(f"Movie ratings.json shape: {movie_ratings_df.count()} rows, {len(movie_ratings_df.columns)} columns")
    movie_ratings_df.show(5)
except Exception as e:
    print(f"Error loading ratings.json: {e}")
    movie_ratings_df = None

Matching.csv shape: 1343 rows, 2 columns


Book tagdl.csv shape: 6814898 rows, 3 columns


Movie tagdl.csv shape: 10551655 rows, 3 columns


Metadata shape: 84661 rows, 6 columns
+---------+---------------+-------+-------+--------------------+--------------------+
|avgRating|     directedBy| imdbId|item_id|            starring|               title|
+---------+---------------+-------+-------+--------------------+--------------------+
|  3.89146|  John Lasseter|0114709|      1|Tim Allen, Tom Ha...|    Toy Story (1995)|
|  3.26605|   Joe Johnston|0113497|      2|Jonathan Hyde, Br...|      Jumanji (1995)|
|  3.17146|  Howard Deutch|0113228|      3|Jack Lemmon, Walt...|Grumpier Old Men ...|
|  2.86824|Forest Whitaker|0114885|      4|Angela Bassett, L...|Waiting to Exhale...|
|   3.0762|  Charles Shyer|0113041|      5|Steve Martin, Mar...|Father of the Bri...|
+---------+---------------+-------+-------+--------------------+--------------------+
only showing top 5 rows


Movie ratings.json shape: 28490116 rows, 3 columns
+-------+------+-------+
|item_id|rating|user_id|
+-------+------+-------+
|      5|   3.0| 997206|
|     10|   4.0| 997206|
|     13|   4.0| 997206|
|     17|   5.0| 997206|
|     21|   4.0| 997206|
+-------+------+-------+
only showing top 5 rows


## 1. Filtering book and movie tags

In order to continue working with the data, we first filter through book and movies tags, by taking the common ones from the matching.csv.
We extract unique tags from both the book and movie datasets using `distinct()` to get unique values. The we count and displays the total number of unique tags in each dataset. We then use set intersection to find tags that appear in both book and movie datasets. These common tags represent shared themes between the two domains. To identify domain-specific tags, we calculate tags that exist only in books (`book_tags - movie_tags`) and those that exist only in movies (`movie_tags - book_tags`). Domain-specific tags have to be removed, since we're trying to work cross-domain, using unified tagging patterns.
The output shows counts for all categories (total, common, and domain-specific tags), displaying sorted lists of the domain-specific tags, which helps us understand the overlap between different tags.

In [8]:
# Get unique tags from each dataset
book_tags = set(book_tagdl_df.select("tag").distinct().rdd.map(lambda row: row.tag).collect())
movie_tags = set(movie_tagdl_df.select("tag").distinct().rdd.map(lambda row: row.tag).collect())

print(f"Unique tags in books: {len(book_tags)}")
print(f"Unique tags in movies: {len(movie_tags)}")

# Find common tags (tags that appear in both domains)
common_tags = book_tags.intersection(movie_tags)
print(f"Common tags (appearing in both domains): {len(common_tags)}")

# Find domain-specific tags
book_only_tags = book_tags - movie_tags
movie_only_tags = movie_tags - book_tags
print(f"Book-only tags to remove: {len(book_only_tags)}")
print(f"Movie-only tags to remove: {len(movie_only_tags)}")

print(sorted(list(book_only_tags)))
print(sorted(list(movie_only_tags))) 

Unique tags in books: 727
Unique tags in movies: 1084
Common tags (appearing in both domains): 491
Book-only tags to remove: 236
Movie-only tags to remove: 593
['2000s', '20th century', '21st century', '3 stars', '4 stars', '5 stars', 'abuse', 'adult', 'afghanistan', 'african americans', 'airplanes', 'all time favorites', 'alpha', 'alpha males', 'america', 'american', 'americas', 'angels', 'angst', 'anthology', 'anti heroes', 'anti semitism', 'anti war', 'apocalyptic', 'arc', 'asia', 'assassins', 'autobiography', 'bad boys', 'banned books', 'bdsm', 'beach read', 'bikers', 'billionaires', 'biology', 'bizarre', 'bomb', 'book boyfriends', 'books about books', 'canadian', 'cartoons', 'celebrity', 'chapter book', 'chick lit', "children's books", 'civil rights', 'cliffhanger', 'cloning', 'comic read', 'contemporary', 'contempor\u200bary', 'cozy mystery', 'cults', 'cultural', 'culture', 'dark romance', 'deserts', 'disasters', 'diversity', 'dual pov', 'dysfunctional families', 'economics', 'ed

Next, we apply the previous tag analysis to create filtered datasets that only contain common cross-domain tags. Using `isin()` we filter rows where the tag column matches the common tags list. Then, we save the filtered book and movie tag datasets to separate CSV files with `coalesce(1)`, so that we could merge the result into a single output file. We repeat these operations for filtering the matching.csv as well. The filter is applied to both `book_tag` and `movie_tag` columns, which ensures that any tag relationships only involve tags that exist in both domains.
The result is a cleaner, more focused dataset for cross-domain analysis.

In [9]:
# Filter out domain-specific tags from both tagdl files
book_tagdl_filtered = book_tagdl_df.filter(col("tag").isin(list(common_tags)))
movie_tagdl_filtered = movie_tagdl_df.filter(col("tag").isin(list(common_tags)))

# Save filtered files
try:
    book_tagdl_filtered.coalesce(1).write.mode("overwrite").option("header", "true").csv("output/book_tagdl_filtered")
    movie_tagdl_filtered.coalesce(1).write.mode("overwrite").option("header", "true").csv("output/movie_tagdl_filtered")
    print("Successfully saved filtered tagdl files")
except Exception as e:
    print(f"Error saving filtered tagdl files: {e}")

# Also filter matching.csv to remove rows with domain-specific tags
matching_filtered = matching_df.filter(
    col("book_tag").isin(list(common_tags)) & 
    col("movie_tag").isin(list(common_tags))
)

try:
    matching_filtered.coalesce(1).write.mode("overwrite").option("header", "true").csv("output/matching_filtered")
    print("Successfully saved filtered matching file")
except Exception as e:
    print(f"Error saving filtered matching file: {e}")

Successfully saved filtered tagdl files
Successfully saved filtered matching file


This chunk helps us visually verify that the tags are the same in each dataset.

In [10]:
# Check first few tags in each dataset
print("First 10 tags in book dataset:")
book_tagdl_filtered.select("tag").distinct().orderBy("tag").show(10)

print("First 10 tags in movie dataset:")  
movie_tagdl_filtered.select("tag").distinct().orderBy("tag").show(10)

First 10 tags in book dataset:


+------------+
|         tag|
+------------+
|18th century|
|       1920s|
|       1930s|
|       1950s|
|       1960s|
|       1970s|
|       1980s|
|19th century|
|    abortion|
|      absurd|
+------------+
only showing top 10 rows
First 10 tags in movie dataset:


+------------+
|         tag|
+------------+
|18th century|
|       1920s|
|       1930s|
|       1950s|
|       1960s|
|       1970s|
|       1980s|
|19th century|
|    abortion|
|      absurd|
+------------+
only showing top 10 rows


## 2. Reducing movies dataset by filtering recent movies (1980-2014+) and movie ratings

In order to reduce the size of the movie dataset, we will be filtering it by years. The chosen time period is 1980-2014 onwards.

### Filtering metadata_updated by year

Since we chose to start with filtering the metadata_updated.json, we'll first need to edit the dataset to extract its film release years. The publication years are contained together with the movie names in the column "title". Thus, we separate the year, by putting it in a separate column and removing it from the "title". To do so, we used `r"\((\d{4})\)"` to find 4-digit years enclosed in parentheses, since that was how the years were represented in the "title". After that, the code creates a new "year" column with the extracted year as an integer. If no year pattern is found, the year is set to `None`.
Because we aim to have a clear visual result, we removed the year and parentheses from the original title with the help of `r"^(.+?)\s*\(\d{4}\)"`, which captures everything before the year pattern. Moreover, the `(.+?)` captured the title text and `\s*` deaalt with any whitespace before the parentheses. To remove any left-behind whitespace from the cleaned title, we used `trim()`. And if no year pattern was found, the original title was left unchanged.
This preprocessing step was crucial for our further analysis, since we'll be using the publication years later for filtering other movie datasets.

In [11]:
# Extract years only from parentheses format (YYYY)
metadata_with_year = metadata_df.withColumn(
    "year",
    when(regexp_extract(col("title"), r"\((\d{4})\)", 1) != "",
         regexp_extract(col("title"), r"\((\d{4})\)", 1).cast("int"))
    .otherwise(None)
).withColumn(
    "title",
    when(regexp_extract(col("title"), r"\((\d{4})\)", 1) != "",
         trim(regexp_extract(col("title"), r"^(.+?)\s*\(\d{4}\)", 1)))
    .otherwise(col("title"))
)

Aferwards, we directly filter movies from 1980 onwards, to decrease the size of the dataset, while still keeping its retained information ratio meaningful. 

In [12]:
# Filter for movies from 1980 onwards with extractable years
metadata_filtered = metadata_with_year.filter(
    (col("year").isNotNull()) & (col("year") >= 1980)
)

Finally, we cache the dataset for the future use.

In [13]:
# Cache the final dataset
metadata_filtered.cache()
print(f"\nFinal dataset shape: {metadata_filtered.count()} rows, {len(metadata_filtered.columns)} columns")
metadata_filtered.show(10)


Final dataset shape: 62806 rows, 7 columns
+---------+---------------+-------+-------+--------------------+--------------------+----+
|avgRating|     directedBy| imdbId|item_id|            starring|               title|year|
+---------+---------------+-------+-------+--------------------+--------------------+----+
|  3.89146|  John Lasseter|0114709|      1|Tim Allen, Tom Ha...|           Toy Story|1995|
|  3.26605|   Joe Johnston|0113497|      2|Jonathan Hyde, Br...|             Jumanji|1995|
|  3.17146|  Howard Deutch|0113228|      3|Jack Lemmon, Walt...|    Grumpier Old Men|1995|
|  2.86824|Forest Whitaker|0114885|      4|Angela Bassett, L...|   Waiting to Exhale|1995|
|   3.0762|  Charles Shyer|0113041|      5|Steve Martin, Mar...|Father of the Bri...|1995|
|  3.85549|   Michael Mann|0113277|      6|Robert De Niro, A...|                Heat|1995|
|  3.37244| Sydney Pollack|0114319|      7|Harrison Ford, Gr...|             Sabrina|1995|
|  3.11678|   Peter Hewitt|0112302|      8|Jon

With this code, we compare the original dataset with a filtered version and provide a summary, by counting the total number of movies in the original `metadata_df` and the total number of movies in the filtered `metadata_filtered` dataset. Using the `agg()` with min/max functions, we confirm that the filtering worked correctly by showing the year range 1980-2014+.
This comparison helps us verify that not only was the data reduction reasonable bbut also that the year extraction and filtering from previous steps worked correctly.

In [14]:
# These will show different results:
print("Original dataset:")
print(f"Total movies: {metadata_df.count()}")

print("\nFiltered dataset (1980+):")
print(f"Total movies: {metadata_filtered.count()}")
print(f"Year range: {metadata_filtered.agg({'year': 'min'}).collect()[0][0]} to {metadata_filtered.agg({'year': 'max'}).collect()[0][0]}")

Original dataset:
Total movies: 84661

Filtered dataset (1980+):
Total movies: 62806
Year range: 1980 to 2021


In [15]:
# Saving as CSV with proper formatting
try:
    metadata_filtered.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("quoteAll", "true") \
        .csv("output/movies_metadata_filtered_1980_plus")
    print("Successfully saved filtered metadata to CSV")
except Exception as e:
    print(f"Error saving metadata CSV: {e}")

Successfully saved filtered metadata to CSV


### Filtering movie ratings by item_id from the filtered metadata

At this stage, we intend to filter movie ratings, so that the only item_id's that the file contains are the ones in the filtered metadata.

With this code, we extract the unique movie identifiers from the filtered dataset. We select only the "item_id" column from the filtered metadata dataset and use `distinct()` to get unique movie IDs, so that no duplicates are left. This creates a new DataFrame containing just the unique movie id's. Apart from that, we verify that the IDs are in the expected format by using `show()`.

In [16]:
# Extracting the item_ids from filtered metadata (1980+ movies)
movie_id_1980_plus = metadata_filtered.select("item_id").distinct()
print(f"Movies from 1980+: {movie_id_1980_plus.count()} unique items")
movie_id_1980_plus.show(5)

Movies from 1980+: 62806 unique items
+-------+
|item_id|
+-------+
|     26|
|     29|
|    474|
|   1677|
|   1697|
+-------+
only showing top 5 rows


In the next step, we filter the movie ratings dataset to only include ratings for movies within the 1980+ time range.
Unlike in our previous pre-processing approcahes, here we use a different method. We perform an inner join between the unfiltered ratings dataset and the filtered movie IDs, which we extracted from the metadata_filtered. This ensures that only ratings for movies that exist in the filtered metadata are retained. Then, to remove the duplicate `item_id` column that we get because of the join operation, we use `drop()`.

In [17]:
# Filtering ratings to only include ratings for 1980+ movies
movie_ratings_1980_plus = movie_ratings_df.join(
    movie_id_1980_plus, 
    movie_ratings_df.item_id == movie_id_1980_plus.item_id, 
    "inner"
).drop(movie_id_1980_plus.item_id)

To analyze the output, we counts the total number of ratings in the filtered dataset and calculates the retention rate as a percentage. This way, we can see how much of the original ratings data is preserved after filtering. 
The retention rate is particularly important because it indicates what portion of the ratings is coming from the years we chose. Since our retention rate is quite high (over 80%), we can conclude that we correctly identified our target time period.

In [18]:
print(f"Filtered movie ratings shape: {movie_ratings_1980_plus.count()} rows")
print(f"Retention rate: {movie_ratings_1980_plus.count()/movie_ratings_df.count()*100:.1f}%")

Filtered movie ratings shape: 24954248 rows


Retention rate: 87.6%


This code saves the filtered movie ratings dataset and caches it for faster access in future use. 

In [19]:
# Save the filtered ratings
try:
    movie_ratings_1980_plus.coalesce(1).write.mode("overwrite").option("header", "true").csv("output/movies_ratings_filtered_1980_plus")
    print("Filtered movie ratings saved to output/movies_ratings_filtered_1980_plus")
except Exception as e:
    print(f"Error saving filtered movie ratings: {e}")


Filtered movie ratings saved to output/movies_ratings_filtered_1980_plus


### Filtering movie tagdl by 1980+ movies

This code does the same time-based filtering to the movie tags dataset, so that all datasets are aligned to the same time period. We, first, get unique movie IDs from the already-filtered ratings dataset. Then, we create a list of movies that have both metadata and ratings data from 1980+. Similarly to the previous step, we join the movie tags dataset, already filtered for common tags, with the time-filtered movie IDs. Using the inner join, we keep only tag data for movies that exist in the ratings dataset and later remove the duplicate `item_id` column with `drop()`.

In [20]:
# Get unique item_ids from the filtered movie ratings (1980+ movies)
movie_id_1980_plus = movie_ratings_1980_plus.select("item_id").distinct()
print(f"Unique movie items from 1980+: {movie_id_1980_plus.count()}")

# Filter movie tagdl to only include items that exist in the 1980+ ratings
movie_tagdl_filtered_1980_plus = movie_tagdl_filtered.join(
movie_id_1980_plus,
movie_tagdl_filtered.item_id == movie_id_1980_plus.item_id,
"inner"
).drop(movie_id_1980_plus.item_id)

Unique movie items from 1980+: 50444


To clearly demonstrate how the dataset has changed, we perform an analysis of data reduction. This involves comparing the size before and after filtering and calculating the retention rate to understand how much tag data corresponds to the target time period. This step helped us assess whether the filtering removed too much important data. Thus, we verify that we're creating a time-aligned dataset ready for further cross-domain analysis.

In [21]:
print(f"Original movie tagdl (common tags only): {movie_tagdl_filtered.count()} rows")
print(f"Filtered movie tagdl (1980+ movies only): {movie_tagdl_filtered_1980_plus.count()} rows")
print(f"Retention rate: {movie_tagdl_filtered_1980_plus.count()/movie_tagdl_filtered.count()*100:.1f}%")

# Show sample of filtered data
print("\nSample of filtered movie tagdl (1980+ movies):")
movie_tagdl_filtered_1980_plus.show(10)

Original movie tagdl (common tags only): 4779394 rows


Filtered movie tagdl (1980+ movies only): 3818998 rows


Retention rate: 79.9%

Sample of filtered movie tagdl (1980+ movies):


+----+-------+--------------------+
| tag|item_id|               score|
+----+-------+--------------------+
|aids|     26|         0.005092442|
|aids|     29|        0.0039109588|
|aids|    474|          0.00500077|
|aids|   1677|        0.0018276572|
|aids|   1806|        0.0041607916|
|aids|   2250|        0.0058024526|
|aids|   2453|0.006698131600000001|
|aids|   2509|0.006292730600000001|
|aids|   3091|        0.0044605434|
|aids|   3764|        0.0039370656|
+----+-------+--------------------+
only showing top 10 rows


This code saves and optimizes the filtered movie tags dataset. It ensures that the processed movie tags data is both preserved on disk and optimized in memory for the next stages of model training, concluding our pre-procesing steps.

In [22]:
# Save the filtered movie tagdl
try:
    movie_tagdl_filtered_1980_plus.coalesce(1).write.mode("overwrite").option("header", "true").csv("output/movie_tagdl_filtered_1980_plus")
    print("Filtered movie tagdl saved to output/movie_tagdl_filtered_1980_plus")
except Exception as e:
    print(f"Error saving filtered movie tagdl: {e}")

# Cache for future use
movie_tagdl_filtered_1980_plus.cache()

Filtered movie tagdl saved to output/movie_tagdl_filtered_1980_plus


tag,item_id,score
aids,26,0.005092442
aids,29,0.0039109588
aids,474,0.00500077
aids,1677,0.0018276572
aids,1806,0.0041607916
aids,2250,0.0058024526
aids,2453,0.006698131600000001
aids,2509,0.006292730600000001
aids,3091,0.0044605434
aids,3764,0.0039370656
